In [1]:
!pip -q install -U transformers accelerate huggingface_hub
import torch, transformers
print("transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))   # should say Tesla T4

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
transformers 5.22.0
GPU: Tesla T4


In [2]:
from huggingface_hub import snapshot_download
import zipfile, os, glob, json, re, string, time
import pandas as pd

ROOT = snapshot_download("BoKelvin/SLAKE", repo_type="dataset", local_dir="/content/slake")
with zipfile.ZipFile(f"{ROOT}/imgs.zip") as f:
    f.extractall(ROOT)
IMG_DIR = os.path.dirname(glob.glob(f"{ROOT}/**/xmlab1", recursive=True)[0])

test = pd.read_json(f"{ROOT}/test.json")
test = test[test.q_lang == "en"].reset_index(drop=True)
print(len(test), "test questions")   # expect 1061

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

1061 test questions


In [3]:
from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/medex_v2"
os.makedirs(OUT, exist_ok=True)
PRED_PATH = f"{OUT}/preds_zeroshot_qwen3vl2b.csv"   # saved as it runs, so a disconnect doesn't lose progress

Mounted at /content/drive


In [4]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,          # T4 has no bf16 support
    device_map="cuda",
    attn_implementation="sdpa",   # FlashAttention doesn't run on T4
).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

VRAM used: 4.3 GB


In [5]:
# standard short-answer prompt used in Med-VQA papers; keep it fixed so before/after are comparable
PROMPT = "{q}\nAnswer the question using a single word or phrase."

@torch.inference_mode()
def predict(img_path, question, max_new_tokens=16):
    msgs = [{"role": "user", "content": [
        {"type": "image", "image": img_path},
        {"type": "text", "text": PROMPT.format(q=question)},
    ]}]
    inputs = processor.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)   # greedy = reproducible
    new_tokens = out[:, inputs.input_ids.shape[1]:]                                  # drop the prompt tokens
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

for _, r in test.sample(5, random_state=0).iterrows():
    print(f"Q: {r.question}\n   gt: {r.answer} | pred: {predict(f'{IMG_DIR}/{r.img_name}', r.question)}\n")

Q: What kind of symptoms will the disease on the heart in this image bring to the patient?
   gt: Chest tightness, arrhythmia | pred: chest pain

Q: What is the organ on the top of the body in this image?
   gt: Liver | pred: liver

Q: Do the organs in the image exist in the head?
   gt: Yes | pred: Yes

Q: Which part of the body does this image belong to?
   gt: Abdomen | pred: abdomen

Q: How many kidneys in this image?
   gt: 2 | pred: 2

